# Day 10 EDA & Data Cleaning Report

This notebook documents a full production-style EDA and cleaning pipeline for a raw dataset.

In [ ]:
import io
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', None)

## Phase 1: Data Inspection & Structural Diagnosis
We inspect shape, schema, and summary statistics before making any transformation.

In [ ]:
df_raw = pd.read_csv('raw_dataset.csv')

print('Initial shape:', df_raw.shape)

buffer = io.StringIO()
df_raw.info(buf=buffer)
print('\nDataFrame info():\n')
print(buffer.getvalue())

print('describe().T output:')
display(df_raw.describe(include='all').T)

In [ ]:
expected_numeric = ['age', 'monthly_income', 'experience_years', 'performance_score', 'satisfaction_index']
object_cols = df_raw.select_dtypes(include=['object']).columns.tolist()
print('Object columns:', object_cols)

structural_issues = {}
for col in expected_numeric:
    rogue = df_raw[col].astype(str).str.contains(r'[^0-9.\-]', regex=True, na=False)
    if rogue.any():
        structural_issues[col] = df_raw.loc[rogue, col].unique().tolist()

print('\nStructural syntax issues (numeric columns contaminated with text):')
for col, values in structural_issues.items():
    print(f'- {col}: {values}')

## Phase 2: Rigorous Data Cleaning Pipeline
Order of operations: duplicate management -> type coercion -> null strategy with justification.

In [ ]:
df = df_raw.copy()

dup_before = df.duplicated().sum()
print('Duplicate rows before drop:', dup_before)

df = df.drop_duplicates().copy()
dup_after = df.duplicated().sum()
print('Duplicate rows after drop:', dup_after)

In [ ]:
numeric_cols = ['age', 'monthly_income', 'experience_years', 'performance_score', 'satisfaction_index']
null_before = df.isna().sum()
print('Null count before coercion:')
display(null_before.to_frame('null_count_before'))

for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(r'[^0-9.\-]', '', regex=True)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert percent-like values (example: 72 -> 0.72)
mask = df['satisfaction_index'] > 1
df.loc[mask, 'satisfaction_index'] = df.loc[mask, 'satisfaction_index'] / 100.0

null_after_coercion = df.isna().sum()
print('Null count after coercion:')
display(null_after_coercion.to_frame('null_count_after_coercion'))

### Missing Value Strategy Justification
- Numeric columns use median imputation because distributions are skewed by outliers (median is robust).
- Categorical `department` uses mode imputation to preserve the most likely class label.

In [ ]:
for col in numeric_cols:
    skew_val = df[col].dropna().skew()
    print(f'Skewness before imputation for {col}: {skew_val:.3f}')

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

df['department'] = df['department'].fillna(df['department'].mode()[0])

# Final type formatting
df['age'] = df['age'].round().astype('int64')
df['experience_years'] = df['experience_years'].round().astype('int64')
df['monthly_income'] = df['monthly_income'].round(2)
df['performance_score'] = df['performance_score'].round(2)
df['satisfaction_index'] = df['satisfaction_index'].round(3)

null_after = df.isna().sum()
print('Null count after imputation:')
display(null_after.to_frame('null_count_after_imputation'))
print('Total nulls:', int(df.isna().sum().sum()))

In [ ]:
print('Final dtypes:')
print(df.dtypes)
print('Final shape:', df.shape)
print('Final duplicate count:', df.duplicated().sum())

df.to_csv('cleaned_dataset.csv', index=False)
print('Saved cleaned dataset to cleaned_dataset.csv')
display(df.head())

## Phase 3: Visual & Statistical Analysis
We use univariate, bivariate, and multivariate analysis to identify skewness, outliers, and feature relationships.

In [ ]:
continuous_cols = ['age', 'monthly_income', 'experience_years', 'performance_score', 'satisfaction_index']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    sns.histplot(df[col], kde=True, bins=10, ax=axes[i], color='teal')
    axes[i].set_title(f'{col} distribution')

for j in range(len(continuous_cols), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

print('Skewness by continuous feature:')
display(df[continuous_cols].skew().to_frame('skewness'))

**Observation:** `monthly_income` is right-skewed due to a very high-income tail. Most other numeric features are relatively stable with moderate skew.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='department', y='monthly_income', palette='Set2')
plt.title('Monthly income by department (outlier view)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='experience_years', y='monthly_income', hue='department', s=80)
plt.title('Experience vs income by department')
plt.show()

# IQR-based outlier tagging for monthly_income
q1 = df['monthly_income'].quantile(0.25)
q3 = df['monthly_income'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
outliers_income = df[(df['monthly_income'] < lower) | (df['monthly_income'] > upper)]

print('Monthly_income outlier bounds:', (round(lower, 2), round(upper, 2)))
print('Detected monthly_income outliers:', len(outliers_income))
display(outliers_income)

**Observation:** Outliers are concentrated in `monthly_income` (especially management-level records), which may distort linear models unless handled with capping or transformation.

In [ ]:
corr = df[continuous_cols].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlGnBu', linewidths=0.5)
plt.title('Correlation heatmap of numeric features')
plt.show()

display(corr)

**Observation:** `age`, `experience_years`, and `monthly_income` show strong positive association, indicating potential redundancy depending on model type and regularization strategy.

## Insights for ML Modeling

1. **High-correlation features / multi-collinearity risk:** `age`, `experience_years`, and `monthly_income` are strongly correlated. For linear models, this can increase variance in coefficient estimates and should be monitored (VIF, regularization, or feature reduction).
2. **Columns with severe outliers:** `monthly_income` contains notable right-tail outliers. These should be handled using capping (winsorization), clipping, or log transformation before training outlier-sensitive models.
3. **Structural changes executed for robust `.fit()` readiness:**
   - Removed exact duplicate rows and verified final duplicate count is zero.
   - Coerced numeric-like text to numeric using parsing and `pd.to_numeric(errors='coerce')`.
   - Standardized percentage-like values in `satisfaction_index` to decimal scale.
   - Imputed missing values with distribution-aware strategy (median for numeric, mode for categorical).
   - Exported final clean table as `cleaned_dataset.csv` with valid dtypes and no nulls.